# Exercise 7: Post-Processing - Regridding and Interpolation

In this exercise, you'll learn how to:
- Use server-side interpolation to request data on different grids
- Use client-side regridding with `earthkit.regrid`
- Convert HEALPix data to regular lat-lon grids
- Export data to different formats

## Authentication

In [ ]:
%%capture cap
%run ../desp-authentication.py

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]

## Imports

In [ ]:
import earthkit.data
import earthkit.plots
import earthkit.regrid
import numpy as np
import matplotlib.pyplot as plt
import os

## Live Request Toggle

In [ ]:
LIVE_REQUEST = os.getenv("LIVE_REQUEST", "true").lower() == "true"
LIVE_REQUEST

## Part 1: Server-Side Interpolation

The Climate DT stores data on a HEALPix grid. We can request server-side interpolation to get data on standard grids by adding a `grid` key to the request.

Supported grid types:
- `O` - Octahedral reduced Gaussian (e.g. `O1280`)
- `F` - Regular Gaussian
- `N` - Classic reduced Gaussian
- `H` - HEALPix ring ordering
- `HN` - HEALPix nested ordering
- Regular lat-lon via `[dx, dy]` (not in this request style)

**EXERCISE**: Change the grid from `O2560` to `O1280` (lower resolution) and add an `interpolation` method.

In [ ]:
request_serverside = {
    "activity": "projections",
    "class": "d1",
    "dataset": "climate-dt",
    "date": "20400102",
    "time": "0000",
    "experiment": "SSP3-7.0",
    "expver": "0001",
    "generation": "2",
    "levtype": "sfc",
    "model": "IFS-NEMO",
    "param": "167",
    "realization": "1",
    "resolution": "standard",
    "stream": "clte",
    "type": "fc",
    "grid": "O256",  # EXERCISE: Change to O1280
    # EXERCISE: Add "interpolation": "linear" here
}

## Retrieve Server-Side Interpolated Data

In [ ]:
data_file = "../climate-dt/data/climate-dt-serverside-training.grib"

if LIVE_REQUEST:
    data_ss = earthkit.data.from_source(
        "polytope", "destination-earth", request_serverside,
        address="polytope.mn5.apps.dte.destination-earth.eu",
        stream=False,
    )
    data_ss.to_target("file", data_file)
else:
    data_ss = earthkit.data.from_source("file", data_file)

data_ss.ls()

## Part 2: Client-Side Regridding

Alternatively, we can retrieve data on its native grid and regrid locally using `earthkit.regrid`.

First, let's get native HEALPix data:

In [ ]:
request_native = {
    "activity": "projections",
    "class": "d1",
    "dataset": "climate-dt",
    "date": "20400102",
    "time": "0000",
    "experiment": "SSP3-7.0",
    "expver": "0001",
    "generation": "2",
    "levtype": "sfc",
    "model": "IFS-NEMO",
    "param": "167",
    "realization": "1",
    "resolution": "standard",
    "stream": "clte",
    "type": "fc",
}

data_file_native = "../climate-dt/data/climate-dt-native-training.grib"

if LIVE_REQUEST:
    data_native = earthkit.data.from_source(
        "polytope", "destination-earth", request_native,
        address="polytope.mn5.apps.dte.destination-earth.eu",
        stream=False,
    )
    data_native.to_target("file", data_file_native)
else:
    data_native = earthkit.data.from_source("file", data_file_native)

data_native.ls()

## Regrid to Regular Lat-Lon

**EXERCISE**: Regrid the native data to a 0.5° × 0.5° regular grid (change from 1° to 0.5°).

In [ ]:
# EXERCISE: Change the output grid from [1, 1] to [0.5, 0.5]
out_grid = {"grid": [1, 1]}  # <- CHANGE THIS

data_regridded = earthkit.regrid.interpolate(
    data_native, 
    out_grid=out_grid, 
    method="linear"
)

print(f"Native fields: {len(data_native)}")
print(f"Regridded fields: {len(data_regridded)}")

## Compare Native vs Regridded

Let's plot both to compare. We use separate plots since `earthkit.plots.Map` manages its own figure internally.

In [ ]:
# Plot native HEALPix data
chart1 = earthkit.plots.Map(extent=[-180, 180, -90, 90])
chart1.grid_cells(data_native[0], style="auto")
chart1.coastlines()
chart1.title("Native HEALPix Grid")
chart1.show()

# Plot regridded data
chart2 = earthkit.plots.Map(extent=[-180, 180, -90, 90])
chart2.grid_cells(data_regridded[0], style="auto")
chart2.coastlines()
chart2.title("Regridded to Regular Lat-Lon")
chart2.show()

## Part 3: Convert to xarray

Regridded data converts cleanly to xarray with proper lat/lon coordinates.

In [ ]:
ds = data_regridded.to_xarray()
print(ds)

# Plot using xarray's built-in plotting
ds["2t"].plot(figsize=(12, 5), cmap="RdYlBu_r")
plt.title("2m Temperature (regridded)")
plt.show()

## EXERCISE: Regrid Extremes DT Data

The Extremes DT uses a reduced Gaussian grid (O2560). Try regridding it to a regular 1°×1° grid.

Complete the code below:

In [ ]:
request_extremes = {
    "class": "d1",
    "dataset": "extremes-dt",
    "expver": "0001",
    "stream": "oper",
    "date": "-14",
    "time": "0000",
    "type": "fc",
    "levtype": "sfc",
    "step": "0",
    "param": "167",
}

# EXERCISE: Retrieve the data and regrid it to [1, 1]
# Hints:
#   1. Use earthkit.data.from_source("polytope", "destination-earth", request_extremes, 
#      address="polytope.lumi.apps.dte.destination-earth.eu", stream=False)
#   2. Use earthkit.regrid.interpolate(data, out_grid={"grid": [1, 1]}, method="linear")

# YOUR CODE HERE